# 🧩 Civitai 下载的 LoRA 模型应用

**LoRA（Low-Rank Adaptation）原理**：不改大模型本身，在 UNet 的注意力层旁边加一对低秩矩阵（lora_down + lora_up），只训练这两个小矩阵——一个风格/功能的 LoRA 通常只有几十 MB（对比 SDXL 权重 6.9GB），加载即用。

**本 LoRA：neg4all_bdsqlsz_xl_V91**（Civitai 下载，适配 SDXL）
- 实测内部结构：1680 个张量，**只有 UNet 部分（无文本编码器权重）**
- 它不是风格 LoRA，而是**负面提示增强 LoRA**：让 negative prompt 的负面约束更有效
- 效果：画质更干净、减少肢体崩坏和杂乱瑕疵；**无需触发词**，加载后照常写负面提示即可
- 用同 seed 开/关 LoRA 对比，可以看到瑕疵的差别

请选择内核：**transformers (15-transformers-basics)**

In [ ]:
# 1. 加载 SDXL 管线（与 03 相同：fp16 + CPU offload，8GB 显存峰值约 5.3GB）
import torch
from diffusers import StableDiffusionXLPipeline

MODEL_DIR = r'S:\PythonAILearningCourse\04-ai-painting-sdxl\stable-diffusion-xl-base-1.0'
LORA_PATH = r'S:\PythonAILearningCourse\04-ai-painting-sdxl\Lora\neg4all_bdsqlsz_xl_V91.safetensors'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_DIR,
    dtype=torch.float16,
    variant='fp16',
)
pipe.enable_model_cpu_offload()
print('✅ SDXL 管线加载完成')

In [ ]:
# 2. 加载 LoRA 文件（civitai 下载的 .safetensors 可直接传文件路径）
#    可加第二个参数 adapter_name='my_lora' 给它起名，方便多 LoRA 管理
pipe.load_lora_weights(LORA_PATH)

# 调整 LoRA 强度：0 = 完全关闭，1 = 原始强度，0.5 = 半强度（负面增强类建议 0.5~1.0）
pipe.set_adapters(['default_0'], adapter_weights=[1.0])

# LoRA 开关：pipe.disable_lora() 临时关闭 / pipe.enable_lora() 重新打开（不用重新加载权重）
print('当前 adapters:', pipe.get_list_adapters())

In [ ]:
# 3. 基准图：关闭 LoRA 生成一张（同 seed 稍后对比）
prompt = 'a group of people walking in a busy market street, crowd, candid photo, detailed'
negative_prompt = 'lowres, bad anatomy, bad hands, extra fingers, blurry, watermark, text'
SEED = 7

pipe.disable_lora()   # ← 关闭 LoRA
img_base = pipe(
    prompt=prompt, negative_prompt=negative_prompt,
    num_inference_steps=25, guidance_scale=5.0,
    height=1024, width=1024,
    generator=torch.Generator('cuda').manual_seed(SEED),
).images[0]
img_base.save(r'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\lora_off.png')
img_base

In [ ]:
# 4. 对比图：开启 LoRA，同 seed 同 prompt —— 观察 AI 人物（手、脸、人群细节）更干净
pipe.enable_lora()    # ← 打开 LoRA
img_lora = pipe(
    prompt=prompt, negative_prompt=negative_prompt,
    num_inference_steps=25, guidance_scale=5.0,
    height=1024, width=1024,
    generator=torch.Generator('cuda').manual_seed(SEED),   # 同种子：唯一变量是 LoRA
).images[0]
img_lora.save(r'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\lora_on.png')
img_lora

In [ ]:
# 5. 左右拼图：直观对比 LoRA 开 / 关的效果
from PIL import Image

combo = Image.new('RGB', (1024, 512))
combo.paste(img_base.resize((512, 512)), (0, 0))      # 左：关闭 LoRA
combo.paste(img_lora.resize((512, 512)), (512, 0))    # 右：开启 LoRA
combo.save(r'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\lora_compare.png')
combo